<a href="https://colab.research.google.com/github/kelvin-yg98/Kelvin-Programming/blob/main/GLM_Bike_Renting_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import kagglehub
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import os

# 1. Download and load the Bike Sharing dataset from Kaggle
path = kagglehub.dataset_download("yasserh/bike-sharing-dataset")
csv_path = os.path.join(path, os.listdir(path)[0])
df_bike = pd.read_csv(csv_path)

# 2. Fit Poisson GLM using statsmodels (IRLS)
formula_bike = 'cnt ~ temp + hum + windspeed'
poisson_glm_bike = smf.glm(formula=formula_bike, data=df_bike, family=sm.families.Poisson()).fit()

# 3. Output results
print("=== STATSMODELS POISSON GLM (IRLS) ===")
print(poisson_glm_bike.summary())
print("\nStatsmodels Log-Likelihood:", poisson_glm_bike.llf)

=== STATSMODELS POISSON GLM (IRLS) ===
                 Generalized Linear Model Regression Results                  
Dep. Variable:                    cnt   No. Observations:                  730
Model:                            GLM   Df Residuals:                      726
Model Family:                 Poisson   Df Model:                            3
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.9821e+05
Date:                Fri, 30 Oct 2026   Deviance:                   3.8903e+05
Time:                        11:14:21   Pearson chi2:                 3.74e+05
No. Iterations:                     5   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    

In [3]:
import numpy as np
import scipy.optimize as opt
import scipy.special
import patsy

# 1. Prepare design matrices from the same formula and dataset
y, X = patsy.dmatrices('cnt ~ temp + hum + windspeed', data=df_bike, return_type='dataframe')
y_vals = y.values.ravel()
X_vals = X.values

# 2. Define exact Negative Log-Likelihood function (including gammaln for exact match)
def negative_poisson_log_likelihood_exact(beta, X, y):
    linear_pred = np.dot(X, beta)
    mu = np.exp(linear_pred)
    # Including log-factorial (gammaln) so the score matches statsmodels identically
    log_likelihood = np.sum(y * np.log(mu + 1e-9) - mu - scipy.special.gammaln(y + 1))
    return -log_likelihood

# 3. Run optimizer starting from zero weights
initial_beta = np.zeros(X_vals.shape[1])
mle_bike_exact = opt.minimize(
    negative_poisson_log_likelihood_exact,
    initial_beta,
    args=(X_vals, y_vals),
    method='Nelder-Mead'
)

# 4. Output custom results
print("=== CUSTOM POISSON MLE ENGINE ===")
print("Optimization Success:", mle_bike_exact.success)
print("Custom MLE Coefficients:\n", mle_bike_exact.x)
print("Custom Exact Log-Likelihood:", -mle_bike_exact.fun)

=== CUSTOM POISSON MLE ENGINE ===
Optimization Success: True
Custom MLE Coefficients:
 [ 8.26482600e+00  3.56562601e-02 -6.50573725e-03 -1.64294697e-02]
Custom Exact Log-Likelihood: -198210.2906927142


In [ ]:
"""
=============================================================================
CONCLUSION & MODEL SUMMARY: POISSON REGRESSION (IRLS VS. CUSTOM MLE)
=============================================================================

Overview:
We modeled daily bike rental counts ('cnt') using continuous weather features
(temperature, humidity, and wind speed) from the Kaggle Bike Sharing dataset.

Two Computational Engines Compared:
1. Statsmodels GLM (IRLS Engine):
   - Solved the Poisson regression model using Iteratively Reweighted Least Squares.

2. Custom Python MLE (scipy.optimize Engine):
   - Built from scratch by explicitly programming the Poisson log-likelihood
     function, incorporating the exact log-factorial term (scipy.special.gammaln).

Key Findings & Takeaways:
- Identical Performance: Both methods yielded identical feature coefficients and
  matched log-likelihood scores of -198,210.29 down to the exact decimal.
- Statistical Significance: P-values of 0.000 for all features confirm that
  temperature, humidity, and wind speed have a statistically meaningful
  impact on ridership rather than occurring by random chance.
- Core Insights: Warmer temperatures increase expected daily rentals (~3.57% per °C),
  while higher humidity (-0.65%) and stronger wind speeds (-1.62%) discourage riders.
=============================================================================
"""